# DataLab Cup2: CNN for Object Detection
TEAM 05 | 112021130 黃偉寧 | 112062313 柯俊安 | 112060002 羅詠璿 | 112062308 蔡佳倩

In [ ]:
import os
from dataclasses import dataclass
from typing import Dict, Sequence, Tuple, Optional

import tensorflow as tf
from keras import Model, layers, regularizers
from keras.applications import convnext
import numpy as np
import albumentations as A
import cv2

## Model Design
參考 yolov4 ，模型包括
- Backbone: ConvNeXt-Tiny
- Neck: Bottom-up Path Aggregation
- Head: 三尺度偵測頭

In [ ]:
def build_yolov4(
    input_shape: Tuple[int, int, int],
    num_classes: int,
    *,
    head_config: Optional[YoloHeadConfig] = None,
    name: str = "yolov4",
    backbone_weights: Optional[str] = "imagenet",
    backbone_trainable: bool = False,
) -> Model:
    """
    Construct YOLOv4 model, returning three raw detection tensors.
    """

    inputs = layers.Input(shape=input_shape, name="image")
    route1, route2, x = _convnext_tiny_backbone(
        inputs,
        weights=backbone_weights,
        trainable=backbone_trainable,
    )

    route_spp = x

    x = _conv_bn_act(x, 256, 1, name="pan_reduce1")
    up = _upsample(x, name="pan_up1")
    route2_proj = _conv_bn_act(route2, 256, 1, name="pan_route2_proj")
    x = layers.Concatenate(name="pan_merge1")([route2_proj, up])
    x = _make_five_layer(x, 256, name="pan_five1")
    route_medium = x

    x = _conv_bn_act(x, 128, 1, name="pan_reduce2")
    up = _upsample(x, name="pan_up2")
    route1_proj = _conv_bn_act(route1, 128, 1, name="pan_route1_proj")
    x = layers.Concatenate(name="pan_merge2")([route1_proj, up])
    x = _make_five_layer(x, 128, name="pan_five2")
    route_small = x

    small_branch_output = _make_head(route_small, 128, num_classes, name="detect_small")

    x = _conv_bn_act(route_small, 256, 3, strides=2, name="pan_down1")
    x = layers.Concatenate(name="pan_merge3")([x, route_medium])
    x = _make_five_layer(x, 256, name="pan_five3")
    route_medium = x
    medium_branch_output = _make_head(route_medium, 256, num_classes, name="detect_medium")

    x = _conv_bn_act(route_medium, 512, 3, strides=2, name="pan_down2")
    x = layers.Concatenate(name="pan_merge4")([x, route_spp])
    x = _make_five_layer(x, 512, name="pan_five4")
    large_branch_output = _make_head(x, 512, num_classes, name="detect_large")

    model = Model(inputs=inputs, outputs=[small_branch_output, medium_branch_output, large_branch_output], name=name)
    if head_config is not None:
        setattr(model, "head_config", head_config)
    return model


## 嘗試與改進

### 1. baseline
用上面的 yolov4 model 和 VOC2007 trainset 訓練 75 個 epoch，在 Kaggle 上 public MSE 0.3544

### 2. Data Augmentation
將 training image 隨機丟入以下增強管道：
* HorizontalFlip + Rotate
* RandomBrightnessContrast + HueSaturationValue
* MotionBlur/GaussianBlur/MedianBlur + GaussNoise
* RandomBrightnessContrast + RandomShadow + RandomFog
* HorizontalFlip + RandomBrightnessContrast + GaussNoise + Rotate

In [1]:
def _create_augmentation_pipelines(self):
    """建立多種資料增強管道"""

    # HorizontalFlip + Rotate：水平翻轉與 ±15° 旋轉。
    pipeline1 = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    # RandomBrightnessContrast + HueSaturationValue：亮度/對比與色相/飽和/明度調整。
    pipeline2 = A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.8),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    # MotionBlur/GaussianBlur/MedianBlur + GaussNoise：模糊效果與高斯噪聲。
    pipeline3 = A.Compose([
        A.OneOf([
            A.MotionBlur(blur_limit=5, p=1.0),
            A.GaussianBlur(blur_limit=5, p=1.0),
            A.MedianBlur(blur_limit=5, p=1.0),
        ], p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    # ShiftScaleRotate + Perspective：平移、縮放、旋轉與透視變換。
    pipeline4 = A.Compose([
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=10,
                            border_mode=cv2.BORDER_CONSTANT, p=0.7),
        A.Perspective(scale=(0.05, 0.1), p=0.3),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    # RandomBrightnessContrast + RandomShadow + RandomFog：天氣與光照效果。
    pipeline5 = A.Compose([
        A.RandomBrightnessContrast(p=0.5),
        A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), num_shadows_lower=1,
                        num_shadows_upper=2, shadow_dimension=5, p=0.3),
        A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, alpha_coef=0.1, p=0.2),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    # HorizontalFlip + RandomBrightnessContrast + GaussNoise + Rotate：綜合增強。
    pipeline6 = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.GaussNoise(var_limit=(10.0, 30.0), p=0.2),
        A.Rotate(limit=10, p=0.3, border_mode=cv2.BORDER_CONSTANT),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    return [pipeline1, pipeline2, pipeline3, pipeline4, pipeline5, pipeline6]

加入 data augmentation 之後變得難 train 了不少，train 了 230 個 epoch loss 才降到 1.66 左右

但 performance 有所提升， kaggle public MSE = 0.23865

### baseline vs data augmentation observation
在原始模型中，baseline 對物件偵測的表現相對薄弱。即使面對單一目標物，模型的 confidence score 普遍偏低。這種「保守式」預測雖然使得模型較不會誤將物件分類至錯誤類別，也降低了多餘 bounding boxes 的產生，因此在低 confidence threshold 下可以有效接收所有預測框，使得 False Positive 偏低、但 Recall 亦受到壓抑。

經過資料增強（data augmentation）後，模型的偵測能力有明顯提升。單一物件的 confidence score 大多接近 1，顯示模型對特徵的擬合度改善。然而，這同時帶出新的模型行為：

* 在類別不平衡情境下，模型會對頻率較高的類別產生偏移，例如在同一張影像中同時偵測出 bus = 1 與 car = 1。
* 當影像中連續出現多個 car 時，模型有時會額外產生第三個 bounding box，顯示模型對相鄰物件的區分度提高，但也開始引入潛在的 over-detection。

  這些現象反映出模型的 Recall 增加，但也伴隨 False Positive 成長的風險。

在量化指標上，資料增強後的模型於 validation set 上的 mean squared error (MSE) 自 baseline 的 0.3544 降至 0.2386，顯示模型在 bounding box regression 上的預測誤差明顯下降。

此外，我注意到模型在處理小物件（例如 bottle、sofa）以及遠距離的物件（如小型飛機或船隻）時，其偵測率明顯較低，這些目標物往往被模型視為背景而遭到忽略。此現象反映出模型在處理 high-frequency, low-resolution features 時的學習能力不足。由於這些物件在資料中佔比低、尺寸又小，baseline 與資料增強後的模型仍無法有效將其特徵從背景中區分出來。

### 3. Focal Loss & Class Balance Weighting

為改善這類物件的偵測能力，並同時處理先前觀察到的類別不平衡與 over-detection 問題，我針對 YOLO loss（由 CIOU loss、objectness loss、class loss 組成）進行調整：

1. Objectness Loss：導入 Focal Loss (α, γ)

原始 objectness loss 以 sigmoid cross-entropy 為主，由於負樣本（背景）密度極高，其梯度會壓過正樣本，使得模型難以從稀疏的小物件中學習到有效特徵。因此，我在 objectness 分支引入 Focal Loss，透過兩個調節係數來重塑權重分布：
* $\alpha$ (alpha factor)：在樣本分布極度不均衡時，用以增強正樣本（物件）的影響力。
* $\gamma$ (gamma modulating factor)：抑制模型對易分類樣本的損失貢獻，使梯度集中於 hard positive，例如小物件、遠距離物件等。

Focal Loss 的形式可寫為：

$$FL(p_t) = -\alpha (1 - p_t)^\gamma \log(p_t)$$

其中：
* $p_t = p$ (若標註為 1)，或 $p_t = 1 - p$ (若標註為 0)，
* $p = \sigma(\text{logit})$ 為模型對 objectness 的預測機率。

設計目的此設計使 objectness loss 更聚焦在低置信度但重要的正樣本，從而改善小物件與遠距離物件「被視為背景」的問題，同時有效提升 Recall。

2. Class Loss：引入 Class-Balanced Weighting

在類別分布極不均衡的物件偵測任務中，出現頻率高的類別（如 **car**）在訓練中往往主導梯度，而低頻類別（如 **bottle**、**sofa**）的貢獻則相對微弱。為改善這一問題，我在 **class 分支**中引入 **Class-Balanced Loss (CB Loss)** (Cui et al., CVPR 2019)，透過「**有效樣本數 (effective number of samples)**」重新定義類別權重，使罕見類別在梯度中具有更高的影響。

2.1 有效樣本數 (Effective Number of Samples)

對於每一個類別 $j$，其樣本數為 $n_j$，則其「有效樣本數」定義為：

$$E(n_j) = \frac{1 - \beta^{n_j}}{1 - \beta}$$

其中：

* $\beta \in [0, 1]$
* $\beta \to 1$ 時，此方法會加強少數類別的權重（常用設定：$0.999 \sim 0.9999$）。

2.2 類別權重 $w_j$

有效樣本數越小（少數類別），其權重應越高，因此採用反比關係：

$$w_j = \frac{1 - \beta}{1 - \beta^{n_j}}$$

In [ ]:
classes_counts = {
    "person": 5447, "car": 1644, "chair": 1432, "bottle": 634, "pottedplant": 625, "bird": 599, 
    "dog": 538, "sofa": 425, "bicycle": 418, "horse": 406, "boat": 398, "motorbike": 390, "cat": 389, 
    "tvmonitor": 367, "cow": 356, "sheep": 353, "aeroplane": 331, "train": 328, "diningtable": 310, "bus": 272
}
# total box: 15662

# ----------------------------------------------------------------------
# 1. Focal Loss for Objectness (處理正負樣本不平衡)
# ----------------------------------------------------------------------

# 計算傳統的 sigmoid cross-entropy (CE)
obj_ce = tf.nn.sigmoid_cross_entropy_with_logits(labels=object_mask, logits=conv_raw_obj)

if self.use_focal:
    # 預測機率 p (objectness score)
    pred_obj = tf.sigmoid(conv_raw_obj)
    
    # 設置 alpha (權重，用於處理正/負樣本數量不均)
    alpha = 0.5 if self.focal_alpha is None else self.focal_alpha
    
    # 計算 alpha_factor: 針對正樣本 (object_mask=1) 應用 alpha，針對負樣本應用 (1-alpha)
    alpha_factor = object_mask * alpha + (1.0 - object_mask) * (1.0 - alpha)
    
    # 計算 p_t: 若為正樣本，p_t=p；若為負樣本，p_t=1-p
    p_t = object_mask * pred_obj + (1.0 - object_mask) * (1.0 - pred_obj)
    p_t = tf.clip_by_value(p_t, EPS, 1.0 - EPS)
    
    # 計算 modulating_factor: (1 - p_t)^gamma (gamma 用於抑制易分類樣本)
    modulating_factor = tf.pow(tf.maximum(1.0 - p_t, EPS), self.focal_gamma)
    
    # Focal Factor = Alpha Factor * Modulating Factor
    focal_factor = alpha_factor * modulating_factor
else:
    focal_factor = 1.0

# 定義 Objectness 權重：
# 1. object_mask: 對正樣本 (True Object) 權重為 1
# 2. (1.0 - object_mask) * ignore_mask: 對負樣本 (Background/Ignore) 權重為 ignore_mask
# obj_ce 乘以 obj_weight 實現了只計算正樣本和 Ignored 負樣本的 CE
obj_weight = object_mask + (1.0 - object_mask) * ignore_mask

# 最終的 Objectness Loss: 應用權重和 Focal Factor
obj_loss = obj_weight * focal_factor * obj_ce


# ----------------------------------------------------------------------
# 2. Class-Balanced Loss (CB Loss) for Classification (處理類別不平衡)
# ----------------------------------------------------------------------

# 計算 Classification Cross-Entropy
cls_ce = tf.nn.sigmoid_cross_entropy_with_logits(labels=true_class, logits=conv_raw_prob)

if self.use_class_balancing:
    # 載入 Class Weights，形狀調整為 (1, 1, 1, 1, num_classes)
    class_weights = tf.reshape(self.class_weights, (1, 1, 1, 1, self.num_classes))
    
    # 計算 CB Factor：只對是該類別的正樣本 (true_class_hard = 1) 應用 class_weights 讓少數類別的正樣本會被賦予更大的權重
    cb_factor = true_class_hard * class_weights + (1.0 - true_class_hard)
else:
    cb_factor = 1.0

# 最終的 Classification Loss: 只計算正樣本 (object_mask)，並應用 CB Factor
cls_loss = object_mask * cb_factor * cls_ce
cls_loss = tf.reduce_sum(cls_loss, axis=-1)

以下是 beta = 0.999 對於 CB Factor / 不同類別 class loss 的影響程度

| 類別 (Class) | 樣本數 (參考 $n_j$) | CB Factor (權重放大倍數 $w_j$) | 
| :---- | :---: | :---: | 
| `person` | 5447 | **1.00** |
| `car` | 1644 | 1.23 |
| `chair` | 1432 | 1.31 |
| `bottle` | 634 | 2.12 |
| `pottedplant` | 625 | 2.14 |
| `bird` | 599 | 2.21 |
| `dog` | 538 | 2.39 |
| `sofa` | 425 | 2.87 |
| `bicycle` | 418 | 2.91 |
| `horse` | 406 | 2.98 |
| `boat` | 398 | 3.03 |
| `motorbike` | 390 | 3.08 |
| `cat` | 389 | 3.09 |
| `tvmonitor` | 367 | 3.24 |
| `cow` | 356 | 3.32 |
| `sheep` | 353 | 3.35 |
| `aeroplane` | 331 | 3.53 |
| `train` | 328 | 3.56 |
| `diningtable` | 310 | 3.73 |
| **`bus`** | 272 | **4.18** |

### 4. Focal CB Train
FocalCB Epoch 240: loss 0.5930
* FOCAL_GAMMA = 1.5        # Focal loss gamma for objectness branch
* FOCAL_ALPHA = None       # None => balanced alpha (0.5)
* CB_BETA = 0.999          # Cui et al. effective-number beta

因為 loss 的計算方法有變，所以不能根據前面的經驗法則來評判這個 model 到底 train 好了沒，但實際丟到 Kaggle 上 public MSE = 0.22482，看起來效用不是很大

實際把 model output visualize 之後，我發現使用 focal-CB 之後 bounding box 與原本相比可以說是稀爛，推斷可能是引入 focal loss、CB loss 之後，比起把 bounding box 畫好，這個 loss 定義讓模型更傾向於把 obj, class loss 降低

我的解決方法稍微回退一點 checkpoint，然後把 CIOU loss 加重加權
```py
ciou_loss *= 4.0
```

這裡稍微岔個題，訓練到後面 LR 不能設太大不然會 learn 不起來。這裡我用了 Consine Decay

In [ ]:
class WarmupCosineSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_start_lr, warmup_steps, total_steps, alpha=0.1):
        super().__init__()
        self.base_lr = base_lr
        self.warmup_start_lr = warmup_start_lr
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.total_steps = total_steps
        self.alpha = alpha
        self.cosine = tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=base_lr,
            decay_steps=(total_steps - warmup_steps),
            alpha=alpha,
        )

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.warmup_start_lr + (self.base_lr - self.warmup_start_lr) * (step / self.warmup_steps)
        cosine_lr = self.cosine(step - self.warmup_steps)

        return tf.cond(step < self.warmup_steps, lambda: warmup_lr, lambda: cosine_lr)

steps_per_epoch = tf.data.experimental.cardinality(dataset).numpy()
total_steps = EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch
lr_schedule = WarmupCosineSchedule(
    base_lr=BASE_LR,
    warmup_start_lr=WARMUP_START_LR,
    warmup_steps=warmup_steps,
    total_steps=total_steps,
    alpha=0.15,  # final lr = 10% base lr
)

在經過額外 200 個 epoch 的匍匐前進之後，public MSE 成功上升到 0.12298

實際觀察 model behavior，儘管他確實在小物件 eg. bottle 或遠方小船的偵測能力有所加強（原本通常會被當成背景忽略掉），且不太容易會有犬馬不分、鳥馬不分這種比較離譜的分類錯誤（car, bus 可能目前還是無解），但模型變得非常「不自信」，對於正樣本的 confidence 明顯過低。

進一步把預測結果攤開來看，confidence 的問題其實不是單純「偏低」而已，而是整個分布變得很不好判斷。大致上可以分成兩種狀況：

1. 只要 confidence 破 0.4，大多就是可靠的正樣本，這部分品質其實不錯。
2. 問題在 0.4 以下 —— 這一段區間裡混著大量的正樣本和負樣本，沒有任何明顯層次。
不會像一般模型那樣，背景會集中在更低分（例如 0.0～0.1），正樣本則會慢慢往中高分靠近。這模型給的感覺比較像：「不是很確定，但先全部丟在這裡」。

所以雖然最終定位品質（MSE）有明顯改善，分類混淆也變少，但 score calibration 完全沒有跟著進步。
threshold 只要稍微調高，小物件就會瞬間消失；threshold 調低，又會引入大量雜訊。對實務來說，這種「中段分數缺乏可讀性」才是最麻煩的部分。

總結起來，Focal + CB 確實把模型的一些弱點補起來了，但也把 score 打得太扁，導致整體 confidence 缺乏細緻度。雖然不至於影響比賽的主要指標（MSE），但在實際使用上就會變得比較難調。

## 附錄：系統架構與程式流程

- `constants.py`：IMAGE SIZE、batch、VOC 類別統計等核心超參數。
- `trainset.py`：`DatasetGenerator` + Albumentations pipelines（透過 `apply_aug` 控制 6 組 augmentation pipelines）。
- `yolov4/`：模型架構與 loss 定義。
- `train.py`：主要訓練腳本，預設 `apply_aug=[True, True, True, False, True, True]`。
- `inference.py`：視覺化腳本，提供約 50 張樣本影像檢閱模型輸出。
- `generate_predictions.py`：批次 inference，輸出 `dataset/test_prediction.txt` 供提交。

## constants.py

In [ ]:
"""
Configuration constants for YOLOv1 model training and evaluation.
"""

import os
from typing import List

ROOT = r'./dataset/'

# common params
IMAGE_SIZE = 416
BATCH_SIZE = 16
NUM_CLASSES = 20
MAX_OBJECTS_PER_IMAGE = 20

# dataset params
DATA_PATH = ROOT + r'pascal_voc_training_data.txt'
IMAGE_DIR = ROOT + r'VOCdevkit_train/VOC2007/JPEGImages/'

# augmentation params
AUGMENT_PROBABILITY = 0.35   # chance of running any Albumentations pipeline per sample
STRETCH_PROBABILITY = 0.25   # chance of running geometric stretch instead of Albumentations

# model params
CELL_SIZE = 7
BOXES_PER_CELL = 2
OBJECT_SCALE = 1
NOOBJECT_SCALE = 0.5
CLASS_SCALE = 1
COORD_SCALE = 5

# Focal / Reweight config
FOCAL_GAMMA = 1.5        # Focal loss gamma for objectness branch
FOCAL_ALPHA = None       # None => balanced alpha (0.5)
CB_BETA = 0.999          # Cui et al. effective-number beta
EPS   = 1e-7             # 避免 log(0)
USE_FOCAL = True         # 是否對 objectness 應用 focal loss
USE_CLASS_BALANCE = True # 是否套用 class-balanced reweighting (classification)

# training params
EPOCHS = 300
WARMUP_EPOCHS = 8
BASE_LR = 2e-4
WARMUP_START_LR = 5e-5


classes_name =  ["aeroplane", "bicycle", "bird", "boat", "bottle",
                 "bus", "car", "cat", "chair", "cow", "diningtable",
                 "dog", "horse", "motorbike", "person", "pottedplant",
                 "sheep", "sofa", "train","tvmonitor"]

classes_counts = {
    "person": 5447, "car": 1644, "chair": 1432, "bottle": 634, "pottedplant": 625, "bird": 599, 
    "dog": 538, "sofa": 425, "bicycle": 418, "horse": 406, "boat": 398, "motorbike": 390, "cat": 389, 
    "tvmonitor": 367, "cow": 356, "sheep": 353, "aeroplane": 331, "train": 328, "diningtable": 310, "bus": 272
}
# total box: 15662

CLASS_COUNTS = [classes_counts[name] for name in classes_name]

VALID_IMAGE_NUMBER: List[int] = [
    936,    # aeroplane
    1579,   # bicycle
    1597,   # bird
    862,    # boat
    4671,   # bottle 
    1892,   # bus 
    9205,   # car 
    1110,   # cat
    830,    # chair 
    464,    # cow
    830,    # diningtable
    4178,   # dog
    2804,   # horse
    1617,   # motorbike
    1101,   # person
    2598,   # pottedplant
    3161,   # sheep 
    2305,   # sofa
    863,    # train
    598     # tvmonitor
]

TEST_BATCH_SIZE = 32
SCORE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.4
MAX_DETECTIONS = 50
TEST_LIST = os.path.join(ROOT, "pascal_voc_testing_data.txt")
TEST_IMAGE_DIR = os.path.join(ROOT, "VOCdevkit_test", "VOC2007", "JPEGImages")
# CHECKPOINT_PATH = os.path.join("ckpts", "yolov4", "ckpt-75")
CHECKPOINT_PATH = os.path.join("ckpts", "focal2", "ckpt-150")
PREDICTION_OPATH = os.path.join("dataset", "test_prediction_o.txt")
FORCE_ITEX_CPU = False  # Set True only when running on Intel XPU/ITEX boxes.


## trainset.py

In [ ]:
from constants import *
import tensorflow as tf
import numpy as np
import albumentations as A
import cv2
import random

class DatasetGenerator:
    """
    Load pascalVOC 2007 dataset and creates an input pipeline.
    - Reshapes images into 416 x 416
    - converts [0 1] to [-1 1]
    - shuffles the input
    - builds batches
    """

    def __init__(self, apply_aug=None, split="train", aug_probability=None, stretch_probability=None):
        self.split = split.lower()
        if self.split not in ("train", "val"):
            raise ValueError(f"Unsupported split '{split}', expected 'train' or 'val'.")

        self.validation_filenames = self._build_validation_filename_set()
        self._storage = {
            "train": {"image_names": [], "record_list": [], "object_num_list": []},
            "val":   {"image_names": [], "record_list": [], "object_num_list": []},
        }

        self.augmentation_pipelines = self._create_augmentation_pipelines()
        self.apply_aug = self._prepare_aug_flags(apply_aug)
        self.active_augmentations = [
            pipeline for flag, pipeline in zip(self.apply_aug, self.augmentation_pipelines) if flag
        ]
        if aug_probability is None:
            self.aug_probability = float(AUGMENT_PROBABILITY)
        else:
            self.aug_probability = float(aug_probability)
        if stretch_probability is None:
            self.stretch_probability = float(STRETCH_PROBABILITY)
        else:
            self.stretch_probability = float(stretch_probability)
        self.aug_probability = min(max(self.aug_probability, 0.0), 1.0)
        self.stretch_probability = min(max(self.stretch_probability, 0.0), 1.0)
        self._read_dataset_file()

        split_bucket = self._storage[self.split]
        self.image_names = split_bucket["image_names"]
        self.record_list = split_bucket["record_list"]
        self.object_num_list = split_bucket["object_num_list"]
        self._drop_warned = True

    def _build_validation_filename_set(self):
        """Convert the manually curated VALID_IMAGE_NUMBER list into filenames."""
        filenames = set()
        for entry in VALID_IMAGE_NUMBER:
            if isinstance(entry, str) and entry.endswith(".jpg"):
                filenames.add(entry)
                continue
            try:
                idx = int(entry)
            except (TypeError, ValueError):
                continue
            filenames.add(f"{idx:06d}.jpg")
        print("filenames:", filenames)
        return filenames

    def _read_dataset_file(self):
        """Populate train/validation buckets according to the curated split."""
        with open(DATA_PATH, 'r') as input_file:
            for line in input_file:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(' ')
                image_name = parts[0]
                record, object_count = self._prepare_record(parts[1:])
                target_split = "val" if image_name in self.validation_filenames else "train"
                bucket = self._storage[target_split]
                bucket["image_names"].append(image_name)
                bucket["record_list"].append(record)
                bucket["object_num_list"].append(object_count)

    def _prepare_record(self, record_tokens):
        """Normalize raw record tokens into a padded list and object count."""
        record = [float(num) for num in record_tokens]
        object_count = min(len(record) // 5, MAX_OBJECTS_PER_IMAGE)
        required_length = MAX_OBJECTS_PER_IMAGE * 5
        if len(record) < required_length:
            # Pad when objects are fewer than MAX_OBJECTS_PER_IMAGE
            padding = [0., 0., 0., 0., 0.] * (MAX_OBJECTS_PER_IMAGE - len(record) // 5)
            record = record + padding
        elif len(record) > required_length:
            # Crop when objects exceed MAX_OBJECTS_PER_IMAGE
            record = record[:required_length]
        return record, object_count

    def _prepare_aug_flags(self, apply_aug):
        """Normalize the augmentation toggle list."""
        pipeline_count = len(self.augmentation_pipelines)
        if apply_aug is None:
            return [False] * pipeline_count
        if isinstance(apply_aug, bool):
            return [apply_aug] * pipeline_count
        if len(apply_aug) != pipeline_count:
            raise ValueError(f"apply_aug must have {pipeline_count} flags, got {len(apply_aug)}")
        return list(apply_aug)

    def _create_augmentation_pipelines(self):
        """建立多種資料增強管道"""

        # HorizontalFlip + Rotate：水平翻轉與 ±15° 旋轉。
        pipeline1 = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

        # RandomBrightnessContrast + HueSaturationValue：亮度/對比與色相/飽和/明度調整。
        pipeline2 = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.8),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

        # MotionBlur/GaussianBlur/MedianBlur + GaussNoise：模糊效果與高斯噪聲。
        pipeline3 = A.Compose([
            A.OneOf([
                A.MotionBlur(blur_limit=5, p=1.0),
                A.GaussianBlur(blur_limit=5, p=1.0),
                A.MedianBlur(blur_limit=5, p=1.0),
            ], p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

        # ShiftScaleRotate + Perspective：平移、縮放、旋轉與透視變換。
        pipeline4 = A.Compose([
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=10,
                              border_mode=cv2.BORDER_CONSTANT, p=0.7),
            A.Perspective(scale=(0.05, 0.1), p=0.3),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

        # RandomBrightnessContrast + RandomShadow + RandomFog：天氣與光照效果。
        pipeline5 = A.Compose([
            A.RandomBrightnessContrast(p=0.5),
            A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), num_shadows_lower=1,
                          num_shadows_upper=2, shadow_dimension=5, p=0.3),
            A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, alpha_coef=0.1, p=0.2),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

        # HorizontalFlip + RandomBrightnessContrast + GaussNoise + Rotate：綜合增強。
        pipeline6 = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
            A.GaussNoise(var_limit=(10.0, 30.0), p=0.2),
            A.Rotate(limit=10, p=0.3, border_mode=cv2.BORDER_CONSTANT),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

        return [pipeline1, pipeline2, pipeline3, pipeline4, pipeline5, pipeline6]

    def _apply_albumentations_np(self, image, labels, object_num):
        """Apply a random enabled augmentation pipeline on numpy arrays."""
        if not self.active_augmentations:
            return image, labels, object_num

        max_boxes = labels.shape[0]
        valid_count = int(object_num)
        selected_pipeline = random.choice(self.active_augmentations)

        bbox_list = []
        class_labels = []
        for idx in range(valid_count):
            xmin, ymin, xmax, ymax, cls = labels[idx]
            if xmax <= xmin or ymax <= ymin:
                continue
            bbox_list.append([xmin, ymin, xmax, ymax])
            class_labels.append(int(cls))

        augmented = selected_pipeline(image=image, bboxes=bbox_list, class_labels=class_labels)
        aug_image = augmented["image"]
        aug_bboxes = augmented["bboxes"]
        aug_classes = augmented["class_labels"]

        padded_labels = np.zeros((max_boxes, 5), dtype=np.float32)
        new_count = min(len(aug_bboxes), max_boxes)
        for idx in range(new_count):
            xmin, ymin, xmax, ymax = aug_bboxes[idx]
            padded_labels[idx] = [xmin, ymin, xmax, ymax, aug_classes[idx]]

        return aug_image.astype(np.uint8), padded_labels.astype(np.float32), np.int32(new_count)

    def _maybe_apply_augmentation(self, image, labels, object_num):
        if self.split == "val":
            return image, labels, object_num

        image = tf.ensure_shape(image, [None, None, 3])
        labels = tf.ensure_shape(labels, [MAX_OBJECTS_PER_IMAGE, 5])
        object_num = tf.ensure_shape(object_num, [])

        aug_prob = self.aug_probability if self.active_augmentations else 0.0
        stretch_prob = self.stretch_probability

        if aug_prob <= 0.0 and stretch_prob <= 0.0:
            return image, labels, object_num

        def _apply():
            aug_image, aug_labels, aug_objs = tf.numpy_function(
                self._apply_albumentations_np,
                [image, labels, object_num],
                [tf.uint8, tf.float32, tf.int32],
            )
            aug_image.set_shape([None, None, 3])
            aug_labels.set_shape([MAX_OBJECTS_PER_IMAGE, 5])
            aug_objs.set_shape([])
            return aug_image, aug_labels, aug_objs

        def _apply_stretch():
            return self._apply_stretch_tf(image, labels, object_num)

        if stretch_prob >= 1.0:
            return _apply_stretch()
        if aug_prob >= 1.0:
            return _apply()

        rnd = tf.random.uniform([], dtype=tf.float32)
        stretch_thresh = tf.constant(stretch_prob, dtype=tf.float32)
        aug_thresh = stretch_thresh + tf.constant(aug_prob, dtype=tf.float32)

        def _maybe_albu():
            return tf.cond(rnd < aug_thresh, _apply, lambda: (image, labels, object_num))

        return tf.cond(
            rnd < stretch_thresh,
            _apply_stretch,
            lambda: tf.cond(rnd < aug_thresh, _apply, lambda: (image, labels, object_num)),
        )

    def _apply_stretch_tf(self, image, labels, object_num):
        image = tf.cast(image, tf.float32) / 127.5 - 1.0
        stretched_img, stretched_labels = self._stretch(image, labels)
        stretched_img = tf.clip_by_value(stretched_img, -1.0, 1.0)
        stretched_img = (stretched_img + 1.0) * 127.5
        stretched_img = tf.cast(tf.clip_by_value(stretched_img, 0.0, 255.0), tf.uint8)
        stretched_img.set_shape([IMAGE_SIZE, IMAGE_SIZE, 3])

        stretched_labels = tf.cast(stretched_labels, tf.float32)
        stretched_labels.set_shape([MAX_OBJECTS_PER_IMAGE, 5])
        widths = stretched_labels[:, 2]
        heights = stretched_labels[:, 3]
        valid_mask = tf.logical_and(widths > 0.0, heights > 0.0)
        new_count = tf.reduce_sum(tf.cast(valid_mask, tf.int32))
        new_count = tf.minimum(new_count, MAX_OBJECTS_PER_IMAGE)
        return stretched_img, stretched_labels, new_count

    def _stretch(self, image, labels, stretch_factor_range=0.4):
        """
        Apply anisotropic stretching with padding/cropping to IMAGE_SIZE.
        """

        out_size = IMAGE_SIZE
        image = tf.convert_to_tensor(image, dtype=tf.float32)
        labels = tf.cast(labels, tf.float32)

        height = tf.shape(image)[0]
        width = tf.shape(image)[1]

        rnd = tf.random.uniform([], 0.0, 1.0)

        def horiz():
            return (
                tf.random.uniform([], 1.0 - stretch_factor_range, 1.0 + stretch_factor_range),
                1.0,
            )

        def vert():
            return (
                1.0,
                tf.random.uniform([], 1.0 - stretch_factor_range, 1.0 + stretch_factor_range),
            )

        scale_w, scale_h = tf.cond(rnd < 0.5, horiz, vert)

        new_h = tf.cast(tf.round(tf.cast(height, tf.float32) * scale_h), tf.int32)
        new_w = tf.cast(tf.round(tf.cast(width, tf.float32) * scale_w), tf.int32)
        new_h = tf.maximum(new_h, 1)
        new_w = tf.maximum(new_w, 1)

        stretched = tf.image.resize(image, (new_h, new_w), method="bilinear")

        pad_top = tf.maximum(0, (out_size - new_h) // 2)
        pad_bottom = tf.maximum(0, out_size - new_h - pad_top)
        pad_left = tf.maximum(0, (out_size - new_w) // 2)
        pad_right = tf.maximum(0, out_size - new_w - pad_left)

        padded = tf.pad(
            stretched,
            [[pad_top, pad_bottom], [pad_left, pad_right], [0, 0]],
            constant_values=-1.0,
        )

        crop_top = tf.maximum(0, (new_h - out_size) // 2)
        crop_left = tf.maximum(0, (new_w - out_size) // 2)

        new_img = tf.image.crop_to_bounding_box(
            padded,
            offset_height=crop_top,
            offset_width=crop_left,
            target_height=out_size,
            target_width=out_size,
        )

        xywh = labels[..., :4]
        cls = labels[..., 4:]

        valid_mask = tf.reduce_any(xywh > 0.0, axis=-1)
        valid_idx = tf.where(valid_mask)

        def no_valid():
            return new_img, labels

        def yes_valid():
            idx_flat = tf.reshape(valid_idx, [-1])
            xywh_valid = tf.gather(xywh, idx_flat)
            scale = tf.stack([scale_w, scale_h, scale_w, scale_h])[None, :]
            xywh_scaled = xywh_valid * scale

            shift = tf.cast([pad_left - crop_left, pad_top - crop_top], tf.float32)
            xy = xywh_scaled[..., :2] + shift
            wh = xywh_scaled[..., 2:4]

            xy = tf.clip_by_value(xy, 0.0, tf.cast(out_size, tf.float32))
            wh = tf.maximum(wh, 0.0)
            xywh_transformed = tf.concat([xy, wh], axis=-1)

            xywh_scattered = tf.tensor_scatter_nd_update(
                xywh, tf.expand_dims(idx_flat, 1), xywh_transformed
            )
            new_labels = tf.concat([xywh_scattered, cls], axis=-1)
            return new_img, new_labels

        return tf.cond(tf.size(valid_idx) > 0, yes_valid, no_valid)

    def _data_preprocess(self, image_name, raw_labels, object_num):
        image_file = tf.io.read_file(IMAGE_DIR+image_name)
        image = tf.io.decode_jpeg(image_file, channels=3)
        raw_labels = tf.cast(tf.reshape(raw_labels, [MAX_OBJECTS_PER_IMAGE, 5]), tf.float32)
        image, raw_labels, object_num = self._maybe_apply_augmentation(image, raw_labels, object_num)
        h = tf.shape(image)[0]
        w = tf.shape(image)[1]

        width_ratio  = IMAGE_SIZE * 1.0 / tf.cast(w, tf.float32)
        height_ratio = IMAGE_SIZE * 1.0 / tf.cast(h, tf.float32)

        image = tf.image.resize(image, size=[IMAGE_SIZE, IMAGE_SIZE])
        image = (image/255) * 2 - 1

        xmin = raw_labels[:, 0]
        ymin = raw_labels[:, 1]
        xmax = raw_labels[:, 2]
        ymax = raw_labels[:, 3]
        class_num = raw_labels[:, 4]

        xcenter = (xmin + xmax) * 1.0 / 2.0 * width_ratio
        ycenter = (ymin + ymax) * 1.0 / 2.0 * height_ratio

        box_w = (xmax - xmin) * width_ratio
        box_h = (ymax - ymin) * height_ratio

        labels = tf.stack([xcenter, ycenter, box_w, box_h, class_num], axis=1)

        return image, labels, tf.cast(object_num, tf.int32)

    def generate(self):
        dataset = tf.data.Dataset.from_tensor_slices((self.image_names,
                                                      np.array(self.record_list),
                                                      np.array(self.object_num_list)))
        if self.split == "train":
            dataset = dataset.shuffle(2500)
        dataset = dataset.map(self._data_preprocess,
                              num_parallel_calls = tf.data.experimental.AUTOTUNE)
        total_images = len(self.image_names)
        remainder = total_images % BATCH_SIZE
        if remainder and not self._drop_warned:
            print(f"[warning] Dropping {remainder} samples from {self.split} split (batch size {BATCH_SIZE}).")
            self._drop_warned = True
        dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
        dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)

        return dataset


## yolov4/model.py

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Sequence, Tuple, Optional

import tensorflow as tf
from keras import Model, layers, regularizers
from keras.applications import convnext


__all__ = [
    "YoloHeadConfig",
    "build_yolov4",
    "decode_predictions",
]

L2_REG = regularizers.l2(0.0005)
ANCHORS_PER_SCALE = 3


class Mish(layers.Layer):
    """Functional Mish activation as a reusable layer."""

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        return inputs * tf.math.tanh(tf.math.softplus(inputs))


def _conv_bn_act(
    x: tf.Tensor,
    filters: int,
    kernel_size: int | Tuple[int, int],
    *,
    strides: int = 1,
    activation: str = "leaky",
    use_bn: bool = True,
    name: Optional[str] = None,
) -> tf.Tensor:
    """Convolution → (BatchNorm) → Activation block."""

    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)

    padding = "same"
    if strides > 1:
        x = layers.ZeroPadding2D(((1, 0), (1, 0)), name=f"{name}_pad" if name else None)(x)
        padding = "valid"

    conv = layers.Conv2D(
        filters,
        kernel_size,
        strides=strides,
        padding=padding,
        use_bias=not use_bn,
        kernel_regularizer=L2_REG,
        kernel_initializer=tf.random_normal_initializer(stddev=0.01),
        bias_initializer=tf.constant_initializer(0.0),
        name=f"{name}_conv" if name else None,
    )(x)

    if use_bn:
        conv = layers.BatchNormalization(name=f"{name}_bn" if name else None)(conv)

    if activation is None:
        return conv

    if activation == "leaky":
        return layers.LeakyReLU(alpha=0.1, name=f"{name}_act" if name else None)(conv)
    if activation == "mish":
        return Mish(name=f"{name}_act" if name else None)(conv)

    raise ValueError(f"Unsupported activation '{activation}'")


def _convnext_tiny_backbone(
    inputs: tf.Tensor,
    *,
    weights: Optional[str] = "imagenet",
    trainable: bool = False,
) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
    """
    ConvNeXt-Tiny backbone returning feature maps that mirror YOLOv4 routes.

    Parameters
    ----------
    inputs:
        Keras input tensor used to build the ConvNeXt backbone.
    weights:
        Weights identifier passed to `ConvNeXtTiny`. Defaults to ImageNet
        pretraining; set to `None` to train from scratch.
    trainable:
        Whether the backbone should remain trainable. Default keeps it frozen
        for transfer learning.
    """

    input_height = inputs.shape[1]
    input_width = inputs.shape[2]
    input_channels = inputs.shape[-1]
    if input_height is None or input_width is None or input_channels is None:
        raise ValueError("ConvNeXt backbone requires statically defined input shape.")
    if input_height != input_width:
        raise ValueError("ConvNeXt backbone currently expects square inputs.")

    # Dataset generator outputs images in [-1, 1]; ConvNeXt expects [0, 1].
    x = layers.Rescaling(scale=0.5, offset=0.5, name="convnext_input_rescale")(inputs)

    base = convnext.ConvNeXtTiny(
        include_top=False,
        include_preprocessing=False,
        weights=weights,
        input_tensor=x,
    )
    base.trainable = trainable

    target_strides = (8, 16, 32)
    stride_to_tensor: Dict[int, tf.Tensor] = {}

    for layer in base.layers:
        outputs = layer.output
        if isinstance(outputs, (list, tuple)):
            tensors = outputs
        else:
            tensors = [outputs]

        for tensor in tensors:
            shape = getattr(tensor, "shape", None)
            if shape is None or len(shape) != 4:
                continue

            h = shape[1]
            w = shape[2]
            if h is None or w is None:
                continue

            stride_h = input_height // int(h)
            stride_w = input_width // int(w)
            if stride_h == stride_w and stride_h in target_strides:
                stride_to_tensor[stride_h] = tensor

    missing = [stride for stride in target_strides if stride not in stride_to_tensor]
    if missing:
        raise ValueError(
            f"Unable to locate ConvNeXt feature maps for strides {missing}. "
            "Please verify the Keras ConvNeXt configuration."
        )

    for stride in target_strides:
        tensor = stride_to_tensor[stride]
        shape = tensor.shape
        fh = shape[1]
        fw = shape[2]
        if fh is None or fw is None:
            raise ValueError(
                "ConvNeXt stride detection requires static feature map shapes. "
                "Please regenerate the backbone with a fixed input resolution."
            )
        stride_h = input_height // int(fh)
        stride_w = input_width // int(fw)
        if stride_h != stride or stride_w != stride:
            raise ValueError(
                f"ConvNeXt stride check failed: expected stride {stride}, "
                f"got {stride_h} (height) and {stride_w} (width)."
            )

    return stride_to_tensor[8], stride_to_tensor[16], stride_to_tensor[32]


def _upsample(x: tf.Tensor, name: str) -> tf.Tensor:
    """Nearest neighbor upsampling by factor 2."""

    return layers.UpSampling2D(size=(2, 2), interpolation="nearest", name=name)(x)


def _make_five_layer(x: tf.Tensor, filters: int, name: str) -> tf.Tensor:
    """Sequence of five conv blocks used repeatedly in the neck."""

    x = _conv_bn_act(x, filters, 1, name=f"{name}_1")
    x = _conv_bn_act(x, filters * 2, 3, name=f"{name}_2")
    x = _conv_bn_act(x, filters, 1, name=f"{name}_3")
    x = _conv_bn_act(x, filters * 2, 3, name=f"{name}_4")
    x = _conv_bn_act(x, filters, 1, name=f"{name}_5")
    return x


def _make_head(x: tf.Tensor, filters: int, num_classes: int, name: str) -> tf.Tensor:
    """Final conv head producing raw YOLO predictions for one scale."""

    x = _conv_bn_act(x, filters * 2, 3, name=f"{name}_conv")
    return layers.Conv2D(
        ANCHORS_PER_SCALE * (num_classes + 5),
        1,
        padding="same",
        kernel_regularizer=L2_REG,
        kernel_initializer=tf.random_normal_initializer(stddev=0.01),
        bias_initializer=tf.constant_initializer(0.0),
        name=f"{name}_pred",
    )(x)


@dataclass(frozen=True)
class YoloHeadConfig:
    """Configuration for decoding YOLOv4 heads."""

    anchors: Sequence[Tuple[float, float]]
    strides: Sequence[int]
    xy_scale: Sequence[float]


def build_yolov4(
    input_shape: Tuple[int, int, int],
    num_classes: int,
    *,
    head_config: Optional[YoloHeadConfig] = None,
    name: str = "yolov4",
    backbone_weights: Optional[str] = "imagenet",
    backbone_trainable: bool = False,
) -> Model:
    """
    Construct YOLOv4 model that returns three raw detection tensors.

    Parameters
    ----------
    input_shape:
        Model input resolution (height, width, channels).
    num_classes:
        Number of object categories.
    head_config:
        Optional decoded head configuration kept for parity with downstream
        utilities. When supplied, it is attached to the returned Keras model
        via the `head_config` attribute for easy retrieval.
    name:
        Model name.
    backbone_weights:
        Weights argument forwarded to `ConvNeXtTiny`. Use `"imagenet"` for
        transfer learning or `None` to initialize randomly.
    backbone_trainable:
        Whether ConvNeXt layers remain trainable.
    """

    inputs = layers.Input(shape=input_shape, name="image")
    route1, route2, x = _convnext_tiny_backbone(
        inputs,
        weights=backbone_weights,
        trainable=backbone_trainable,
    )

    route_spp = x

    x = _conv_bn_act(x, 256, 1, name="pan_reduce1")
    up = _upsample(x, name="pan_up1")
    route2_proj = _conv_bn_act(route2, 256, 1, name="pan_route2_proj")
    x = layers.Concatenate(name="pan_merge1")([route2_proj, up])
    x = _make_five_layer(x, 256, name="pan_five1")
    route_medium = x

    x = _conv_bn_act(x, 128, 1, name="pan_reduce2")
    up = _upsample(x, name="pan_up2")
    route1_proj = _conv_bn_act(route1, 128, 1, name="pan_route1_proj")
    x = layers.Concatenate(name="pan_merge2")([route1_proj, up])
    x = _make_five_layer(x, 128, name="pan_five2")
    route_small = x

    small_branch_output = _make_head(route_small, 128, num_classes, name="detect_small")

    x = _conv_bn_act(route_small, 256, 3, strides=2, name="pan_down1")
    x = layers.Concatenate(name="pan_merge3")([x, route_medium])
    x = _make_five_layer(x, 256, name="pan_five3")
    route_medium = x
    medium_branch_output = _make_head(route_medium, 256, num_classes, name="detect_medium")

    x = _conv_bn_act(route_medium, 512, 3, strides=2, name="pan_down2")
    x = layers.Concatenate(name="pan_merge4")([x, route_spp])
    x = _make_five_layer(x, 512, name="pan_five4")
    large_branch_output = _make_head(x, 512, num_classes, name="detect_large")

    model = Model(inputs=inputs, outputs=[small_branch_output, medium_branch_output, large_branch_output], name=name)
    if head_config is not None:
        setattr(model, "head_config", head_config)
    return model


def _reshape_predictions(pred: tf.Tensor) -> tf.Tensor:
    shape = tf.shape(pred)
    grid_h, grid_w = shape[1], shape[2]
    return tf.reshape(
        pred,
        (-1, grid_h, grid_w, ANCHORS_PER_SCALE, pred.shape[-1] // ANCHORS_PER_SCALE),
    )


def decode_predictions(
    pred: tf.Tensor,
    anchors: tf.Tensor,
    stride: int,
    xy_scale: float = 1.0,
) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor, tf.Tensor]:
    """
    Decode raw head outputs into bounding boxes, objectness, and class logits.

    Parameters
    ----------
    pred:
        Raw head tensor `[B, H, W, 3*(5+num_classes)]`.
    anchors:
        Tensor of shape `[3, 2]` containing anchor widths and heights.
    stride:
        Down-sampling stride relative to the input resolution.
    xy_scale:
        Additional scaling factor used by YOLOv4 to stabilize sigmoid outputs.
    """

    pred = _reshape_predictions(pred)
    grid_h = tf.shape(pred)[1]
    grid_w = tf.shape(pred)[2]
    anchors = tf.cast(anchors, tf.float32)
    anchors = tf.reshape(anchors, (1, 1, 1, ANCHORS_PER_SCALE, 2))

    grid_x, grid_y = tf.meshgrid(tf.range(grid_w), tf.range(grid_h))
    xy_grid = tf.stack([grid_x, grid_y], axis=-1)
    xy_grid = tf.cast(tf.reshape(xy_grid, (1, grid_h, grid_w, 1, 2)), tf.float32)

    pred_xy = (
        (tf.sigmoid(pred[..., 0:2]) * xy_scale - 0.5 * (xy_scale - 1.0) + xy_grid) * stride
    )
    pred_wh = tf.exp(pred[..., 2:4]) * anchors * stride

    bbox = tf.concat([pred_xy, pred_wh], axis=-1)
    objectness = tf.sigmoid(pred[..., 4:5])
    class_probs = pred[..., 5:]



## yolov4/loss.py

In [ ]:
"""
YOLOv4 loss functions.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Iterable, List, Sequence, Tuple

import tensorflow as tf

from .model import ANCHORS_PER_SCALE


__all__ = [
    "YoloLoss",
    "build_yolov4_losses",
]

EPS = 1e-7


def _bbox_xywh_to_corners(box: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    """Convert `[x, y, w, h]` to corner coordinates."""

    xy = box[..., 0:2]
    wh = box[..., 2:4]
    min_corner = xy - wh / 2.0
    max_corner = xy + wh / 2.0
    return min_corner, max_corner


def bbox_iou(boxes1: tf.Tensor, boxes2: tf.Tensor) -> tf.Tensor:
    """
    Compute IoU between boxes in center-width-height format.

    `boxes1` and `boxes2` must be broadcastable to the same shape with last
    dimension of 4.
    """

    boxes1_min, boxes1_max = _bbox_xywh_to_corners(boxes1)
    boxes2_min, boxes2_max = _bbox_xywh_to_corners(boxes2)

    intersect_min = tf.maximum(boxes1_min, boxes2_min)
    intersect_max = tf.minimum(boxes1_max, boxes2_max)
    intersect_wh = tf.maximum(intersect_max - intersect_min, 0.0)
    intersect_area = intersect_wh[..., 0] * intersect_wh[..., 1]

    boxes1_wh = boxes1_max - boxes1_min
    boxes2_wh = boxes2_max - boxes2_min
    boxes1_area = boxes1_wh[..., 0] * boxes1_wh[..., 1]
    boxes2_area = boxes2_wh[..., 0] * boxes2_wh[..., 1]

    union = boxes1_area + boxes2_area - intersect_area
    return intersect_area / (union + EPS)


def bbox_ciou(boxes1: tf.Tensor, boxes2: tf.Tensor) -> tf.Tensor:
    """Complete IoU (CIOU) metric for YOLOv4."""

    iou = bbox_iou(boxes1, boxes2)

    boxes1_min, boxes1_max = _bbox_xywh_to_corners(boxes1)
    boxes2_min, boxes2_max = _bbox_xywh_to_corners(boxes2)

    center_dist = tf.reduce_sum(tf.square(boxes1[..., 0:2] - boxes2[..., 0:2]), axis=-1)
    enclose_min = tf.minimum(boxes1_min, boxes2_min)
    enclose_max = tf.maximum(boxes1_max, boxes2_max)
    enclose_wh = tf.maximum(enclose_max - enclose_min, 0.0)
    enclose_diag = tf.reduce_sum(tf.square(enclose_wh), axis=-1)

    boxes1_wh = boxes1[..., 2:4]
    boxes2_wh = boxes2[..., 2:4]
    v = (4.0 / (math.pi ** 2)) * tf.square(
        tf.atan(boxes2_wh[..., 0] / (boxes2_wh[..., 1] + EPS))
        - tf.atan(boxes1_wh[..., 0] / (boxes1_wh[..., 1] + EPS))
    )
    with tf.name_scope("ciou_alpha"):
        alpha = v / (1.0 - iou + v + EPS)

    return iou - (center_dist / (enclose_diag + EPS) + alpha * v)


@dataclass(frozen=True)
class LossConfig:
    num_classes: int
    input_size: int
    anchor: Sequence[Tuple[float, float]]
    stride: int
    xy_scale: float = 1.0
    ignore_threshold: float = 0.5
    label_smoothing: float = 0.0
    focal_gamma: float = 2.0
    focal_alpha: float | None = 0.5
    cb_beta: float = 0.999
    class_counts: Sequence[float] | None = None
    use_focal: bool = False
    use_class_balancing: bool = False


class YoloLoss(tf.keras.losses.Loss):
    """
    Standalone loss for a single YOLOv4 prediction scale.

    The loss expects:
    - `y_pred`: raw network output `[B, H, W, 3 * (5 + num_classes)]`
    - `y_true`: target tensor `[B, H, W, 3, 5 + num_classes]`
               storing center/width/height in input-space pixels, followed by
               the objectness flag and one-hot class vector.
    """

    def __init__(self, config: LossConfig, name: str | None = None):
        super().__init__(reduction=tf.keras.losses.Reduction.NONE, name=name)
        self.config = config
        self.anchors = tf.constant(config.anchor, dtype=tf.float32)
        self.stride = float(config.stride)
        self.xy_scale = float(config.xy_scale)
        self.ignore_threshold = float(config.ignore_threshold)
        self.num_classes = int(config.num_classes)
        self.input_size = float(config.input_size)
        self.label_smoothing = float(config.label_smoothing)
        self.focal_gamma = float(config.focal_gamma)
        self.focal_alpha = None if config.focal_alpha is None else float(config.focal_alpha)
        self.cb_beta = float(config.cb_beta)
        self.use_focal = bool(config.use_focal)
        self.use_class_balancing = bool(config.use_class_balancing)

        if config.class_counts is None:
            class_counts = tf.ones((self.num_classes,), dtype=tf.float32)
        else:
            counts_list = list(config.class_counts)
            if len(counts_list) != self.num_classes:
                raise ValueError("class_counts must be a 1-D sequence with num_classes elements.")
            class_counts = tf.convert_to_tensor(counts_list, dtype=tf.float32)
        self.class_weights = self._build_class_weights(class_counts)

        # ---- Sanity Check ----
        print("Class-balanced weights:")
        print("  min =", tf.reduce_min(self.class_weights).numpy(),
            "max =", tf.reduce_max(self.class_weights).numpy(),
            "mean =", tf.reduce_mean(self.class_weights).numpy())
        print("Focal settings: alpha =", self.focal_alpha, "gamma =", self.focal_gamma)
        print("Using CB:", self.use_class_balancing, "| Using Focal:", self.use_focal)
        # ----------------------

    def _build_class_weights(self, counts: tf.Tensor) -> tf.Tensor:
        """Compute normalized class-balanced weights from raw counts."""

        beta = tf.convert_to_tensor(self.cb_beta, dtype=tf.float32)
        effective_num = 1.0 - tf.pow(beta, counts)
        effective_num = tf.maximum(effective_num, EPS)
        weights = (1.0 - beta) / effective_num
        weights /= tf.reduce_mean(weights)
        return weights

    def call(self, y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
        total, _, _, _ = self._compute_total_and_components(y_true, y_pred)
        return total

    def compute_total_and_components(
        self, y_true: tf.Tensor, y_pred: tf.Tensor
    ) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor, tf.Tensor]:
        """
        Return the total loss and its CIOU, objectness, and classification components.
        """

        return self._compute_total_and_components(y_true, y_pred)

    def _compute_total_and_components(
        self, y_true: tf.Tensor, y_pred: tf.Tensor
    ) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor, tf.Tensor]:
        shape = tf.shape(y_pred)
        batch = shape[0]
        grid_h = shape[1]
        grid_w = shape[2]

        y_pred = tf.reshape(y_pred, (batch, grid_h, grid_w, ANCHORS_PER_SCALE, 5 + self.num_classes))

        conv_raw_xy = y_pred[..., 0:2]
        conv_raw_wh = y_pred[..., 2:4]
        conv_raw_obj = y_pred[..., 4:5]
        conv_raw_prob = y_pred[..., 5:]

        grid_x, grid_y = tf.meshgrid(tf.range(grid_w), tf.range(grid_h))
        xy_grid = tf.stack([grid_x, grid_y], axis=-1)
        xy_grid = tf.cast(tf.reshape(xy_grid, (1, grid_h, grid_w, 1, 2)), tf.float32)

        anchors_tensor = tf.reshape(self.anchors, (1, 1, 1, ANCHORS_PER_SCALE, 2))

        pred_xy = (tf.sigmoid(conv_raw_xy) * self.xy_scale - 0.5 * (self.xy_scale - 1.0) + xy_grid) * self.stride
        pred_wh = tf.exp(conv_raw_wh) * anchors_tensor * self.stride
        pred_box = tf.concat([pred_xy, pred_wh], axis=-1)

        true_xy = y_true[..., 0:2]
        true_wh = y_true[..., 2:4]
        true_box = tf.concat([true_xy, true_wh], axis=-1)
        object_mask = y_true[..., 4:5]
        true_class = y_true[..., 5:]
        true_class_hard = true_class

        bbox_loss_scale = 2.0 - (true_wh[..., 0] * true_wh[..., 1]) / (self.input_size ** 2)
        bbox_loss_scale = tf.expand_dims(bbox_loss_scale, axis=-1)

        ciou = bbox_ciou(pred_box, true_box)
        ciou_loss = object_mask[..., 0] * bbox_loss_scale[..., 0] * (1.0 - ciou)

        pred_box_expanded = tf.expand_dims(pred_box, axis=-2)
        true_box_reshaped = tf.reshape(true_box, (batch, -1, 4))
        true_box_reshaped = tf.expand_dims(true_box_reshaped, axis=1)
        true_box_reshaped = tf.expand_dims(true_box_reshaped, axis=1)
        true_box_reshaped = tf.expand_dims(true_box_reshaped, axis=1)

        true_obj_mask = tf.reshape(object_mask, (batch, -1, 1))
        true_obj_mask = tf.expand_dims(true_obj_mask, axis=1)
        true_obj_mask = tf.expand_dims(true_obj_mask, axis=1)
        true_obj_mask = tf.expand_dims(true_obj_mask, axis=1)

        iou_scores = bbox_iou(pred_box_expanded, true_box_reshaped) * true_obj_mask[..., 0]
        best_iou = tf.reduce_max(iou_scores, axis=-1, keepdims=True)
        ignore_mask = tf.cast(best_iou < self.ignore_threshold, tf.float32)

        obj_ce = tf.nn.sigmoid_cross_entropy_with_logits(labels=object_mask, logits=conv_raw_obj)

        if self.use_focal:
            pred_obj = tf.sigmoid(conv_raw_obj)
            alpha = 0.5 if self.focal_alpha is None else self.focal_alpha
            alpha_factor = object_mask * alpha + (1.0 - object_mask) * (1.0 - alpha)
            p_t = object_mask * pred_obj + (1.0 - object_mask) * (1.0 - pred_obj)
            p_t = tf.clip_by_value(p_t, EPS, 1.0 - EPS)
            modulating_factor = tf.pow(tf.maximum(1.0 - p_t, EPS), self.focal_gamma)
            focal_factor = alpha_factor * modulating_factor
        else:
            focal_factor = 1.0

        obj_weight = object_mask + (1.0 - object_mask) * ignore_mask
        obj_loss = obj_weight * focal_factor * obj_ce

        if self.label_smoothing > 0.0:
            smooth = self.label_smoothing
            num_classes = tf.cast(self.num_classes, tf.float32)
            true_class = true_class * (1.0 - smooth) + smooth / num_classes

        cls_ce = tf.nn.sigmoid_cross_entropy_with_logits(labels=true_class, logits=conv_raw_prob)

        if self.use_class_balancing:
            class_weights = tf.reshape(self.class_weights, (1, 1, 1, 1, self.num_classes))
            cb_factor = true_class_hard * class_weights + (1.0 - true_class_hard)
        else:
            cb_factor = 1.0

        cls_loss = object_mask * cb_factor * cls_ce
        cls_loss = tf.reduce_sum(cls_loss, axis=-1)

        ciou_loss = tf.reduce_sum(ciou_loss, axis=[1, 2, 3])
        obj_loss = tf.reduce_sum(obj_loss, axis=[1, 2, 3, 4])
        cls_loss = tf.reduce_sum(cls_loss, axis=[1, 2, 3])

        ciou_loss *= 4.0
        total = ciou_loss + obj_loss + cls_loss
        return total, ciou_loss, obj_loss, cls_loss


def build_yolov4_losses(
    *,
    num_classes: int,
    input_size: int,
    anchors: Sequence[Sequence[Tuple[float, float]]],
    strides: Sequence[int],
    xy_scales: Sequence[float] | None = None,
    ignore_threshold: float = 0.5,
    label_smoothing: float = 0.0,
    focal_gamma: float = 2.0,
    focal_alpha: float | None = 0.5,
    cb_beta: float = 0.999,
    class_counts: Sequence[float] | None = None,
    use_focal_loss: bool = False,
    use_class_balancing: bool = False,
) -> List[YoloLoss]:
    """
    Convenience factory returning a list of `YoloLoss` objects for the three scales.

    Parameters
    ----------
    num_classes:
        Number of object categories.
    input_size:
        Training input resolution in pixels (assumes square inputs).
    anchors:
        Sequence of anchor sets, one per scale. Each set must contain three `(w, h)` tuples.
    strides:
        Feature map strides relative to the input image, one per scale.
    xy_scales:
        Optional scale factors used by YOLOv4 to adjust sigmoid outputs (defaults to 1.0).
    ignore_threshold:
        IoU threshold above which non-matching predictions are ignored when computing
        negative objectness loss.
    label_smoothing:
        Apply epsilon label smoothing to classification targets.
    focal_gamma / focal_alpha:
        Hyper-parameters for focal loss modulation on the objectness branch (alpha None => 0.5).
    cb_beta:
        Beta term used when computing class-balanced weights.
    class_counts:
        Dataset counts per class in label order. Required when class balancing is enabled.
    use_focal_loss / use_class_balancing:
        Enable the respective reweighting strategies.
    """

    if xy_scales is None:
        xy_scales = [1.0, 1.0, 1.0]

    losses: List[YoloLoss] = []
    for idx, (anchor_set, stride, xy_scale) in enumerate(zip(anchors, strides, xy_scales)):
        config = LossConfig(
            num_classes=num_classes,
            input_size=input_size,
            anchor=anchor_set,
            stride=stride,
            xy_scale=xy_scale,
            ignore_threshold=ignore_threshold,
            label_smoothing=label_smoothing,
            focal_gamma=focal_gamma,
            focal_alpha=focal_alpha,
            cb_beta=cb_beta,
            class_counts=class_counts,
            use_focal=use_focal_loss,
            use_class_balancing=use_class_balancing,
        )
        losses.append(YoloLoss(config, name=f"yolov4_loss_scale_{idx}"))
    return losses


## train.py

In [ ]:
"""
Training script for YOLOv4 model adapted to the competition pipeline.
"""

from __future__ import annotations

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Set TensorFlow logging to only show errors

from datetime import datetime
from tqdm import tqdm
from typing import List

import numpy as np
import tensorflow as tf

from constants import *
from trainset import DatasetGenerator
from yolov4 import build_yolov4, build_yolov4_losses

GPU_CONFIG = 'GPU' # Change to 'GPU' if using NVIDIA GPUs
gpus = tf.config.experimental.list_physical_devices(GPU_CONFIG)
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        # Select GPU number 1
        tf.config.experimental.set_visible_devices(gpus[0], GPU_CONFIG)
        logical_gpus = tf.config.experimental.list_logical_devices(GPU_CONFIG)
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

ANCHOR_SCALE = IMAGE_SIZE / 448.0
ANCHOR_SETS = [
    [(12 * ANCHOR_SCALE, 16 * ANCHOR_SCALE), (19 * ANCHOR_SCALE, 36 * ANCHOR_SCALE), (40 * ANCHOR_SCALE, 28 * ANCHOR_SCALE)],
    [(36 * ANCHOR_SCALE, 75 * ANCHOR_SCALE), (76 * ANCHOR_SCALE, 55 * ANCHOR_SCALE), (72 * ANCHOR_SCALE, 146 * ANCHOR_SCALE)],
    [(142 * ANCHOR_SCALE, 110 * ANCHOR_SCALE), (192 * ANCHOR_SCALE, 243 * ANCHOR_SCALE), (459 * ANCHOR_SCALE, 401 * ANCHOR_SCALE)],
]
STRIDES = [8, 16, 32]
XY_SCALES = [1.2, 1.1, 1.05]
ANCHORS_PER_SCALE = 3

ANCHORS_FLAT = np.array([anchor for group in ANCHOR_SETS for anchor in group], dtype=np.float32)
ANCHOR_AREAS = ANCHORS_FLAT[:, 0] * ANCHORS_FLAT[:, 1]


class WarmupCosineSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_start_lr, warmup_steps, total_steps, alpha=0.1):
        super().__init__()
        self.base_lr = base_lr
        self.warmup_start_lr = warmup_start_lr
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.total_steps = total_steps
        self.alpha = alpha
        self.cosine = tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=base_lr,
            decay_steps=(total_steps - warmup_steps),
            alpha=alpha,
        )

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.warmup_start_lr + (self.base_lr - self.warmup_start_lr) * (step / self.warmup_steps)
        cosine_lr = self.cosine(step - self.warmup_steps)

        return tf.cond(step < self.warmup_steps, lambda: warmup_lr, lambda: cosine_lr)


def _build_targets_np(labels_batch: np.ndarray, counts_batch: np.ndarray) -> List[np.ndarray]:
    """Encode a batch of VOC boxes into three YOLOv4 training targets."""

    batch_size = labels_batch.shape[0]
    counts_batch = counts_batch.astype(np.int32)
    targets: List[np.ndarray] = []
    for stride in STRIDES:
        grid = IMAGE_SIZE // stride
        targets.append(
            np.zeros((batch_size, grid, grid, ANCHORS_PER_SCALE, 5 + NUM_CLASSES), dtype=np.float32)
        )

    for b in range(batch_size):
        num_obj = int(counts_batch[b])
        for obj_idx in range(num_obj):
            x, y, w, h, cls = labels_batch[b, obj_idx]
            if w <= 0 or h <= 0:
                continue
            class_id = int(cls)
            if class_id < 0 or class_id >= NUM_CLASSES:
                continue

            box_area = w * h
            inter_w = np.minimum(w, ANCHORS_FLAT[:, 0])
            inter_h = np.minimum(h, ANCHORS_FLAT[:, 1])
            inter_area = inter_w * inter_h
            iou = inter_area / (box_area + ANCHOR_AREAS - inter_area + 1e-9)
            best_anchor = int(np.argmax(iou))

            scale_idx = best_anchor // ANCHORS_PER_SCALE
            anchor_idx = best_anchor % ANCHORS_PER_SCALE
            stride = STRIDES[scale_idx]
            grid = IMAGE_SIZE // stride

            grid_x = int(np.floor(x / stride))
            grid_y = int(np.floor(y / stride))
            grid_x = np.clip(grid_x, 0, grid - 1)
            grid_y = np.clip(grid_y, 0, grid - 1)

            targets[scale_idx][b, grid_y, grid_x, anchor_idx, 0:4] = [x, y, w, h]
            targets[scale_idx][b, grid_y, grid_x, anchor_idx, 4] = 1.0
            targets[scale_idx][b, grid_y, grid_x, anchor_idx, 5 + class_id] = 1.0

    return targets


def build_targets(labels_batch: tf.Tensor, counts_batch: tf.Tensor) -> List[tf.Tensor]:
    """TensorFlow wrapper that builds targets via tf.numpy_function to stay on-device."""

    target_small, target_medium, target_large = tf.numpy_function(
        func=_build_targets_np,
        inp=[labels_batch, counts_batch],
        Tout=[tf.float32, tf.float32, tf.float32],
    )

    targets_tf = [target_small, target_medium, target_large]
    for target_tensor, stride in zip(targets_tf, STRIDES):
        grid = IMAGE_SIZE // stride
        target_tensor.set_shape([BATCH_SIZE, grid, grid, ANCHORS_PER_SCALE, 5 + NUM_CLASSES])

    return targets_tf


def get_current_lr(optimizer: tf.keras.optimizers.Optimizer) -> tf.Tensor:
    """Return the scalar learning rate for the current optimizer iteration."""

    lr = optimizer.learning_rate
    if isinstance(lr, tf.keras.optimizers.schedules.LearningRateSchedule):
        return tf.cast(lr(optimizer.iterations), tf.float32)
    return tf.cast(lr, tf.float32)


def main() -> None:
    # Apply data augmentation pipelines except pipeline4 (ShiftScaleRotate + Perspective)
    dataset = DatasetGenerator(
        apply_aug = [True, True, True, False, True, True],
        split='train'
    ).generate()

    # This returns a LR for each global step (not per epoch)
    # steps_per_epoch will receive -2 if used .repeat() in dataset
    steps_per_epoch = tf.data.experimental.cardinality(dataset).numpy()
    total_steps = EPOCHS * steps_per_epoch
    warmup_steps = WARMUP_EPOCHS * steps_per_epoch
    lr_schedule = WarmupCosineSchedule(
        base_lr=BASE_LR,
        warmup_start_lr=WARMUP_START_LR,
        warmup_steps=warmup_steps,
        total_steps=total_steps,
        alpha=0.15,  # final lr = 10% base lr
    )

    model = build_yolov4((IMAGE_SIZE, IMAGE_SIZE, 3), num_classes=NUM_CLASSES)
    loss_fns = build_yolov4_losses(
        num_classes=NUM_CLASSES,
        input_size=IMAGE_SIZE,
        anchors=ANCHOR_SETS,
        strides=STRIDES,
        xy_scales=XY_SCALES,
        focal_gamma=FOCAL_GAMMA,
        focal_alpha=FOCAL_ALPHA,
        cb_beta=CB_BETA,
        class_counts=CLASS_COUNTS,
        use_focal_loss=USE_FOCAL,
        use_class_balancing=USE_CLASS_BALANCE,
    )

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
    train_loss_metric = tf.keras.metrics.Mean(name="train_loss")
    ciou_loss_metric = tf.keras.metrics.Mean(name="ciou_loss")
    obj_loss_metric = tf.keras.metrics.Mean(name="obj_loss")
    cls_loss_metric = tf.keras.metrics.Mean(name="cls_loss")

    ckpt = tf.train.Checkpoint(epoch=tf.Variable(0), model=model, optimizer=optimizer)
    manager = tf.train.CheckpointManager(ckpt, "./ckpts/focal2", max_to_keep=30)
    if manager.latest_checkpoint:
        status = ckpt.restore(manager.latest_checkpoint)
        status.expect_partial()
        print(f"Restored checkpoint from {manager.latest_checkpoint}")
        optimizer.learning_rate = lr_schedule
    else:
        print("Starting fresh training run.")

    @tf.function
    def train_step(images, target_small, target_medium, target_large):
        with tf.GradientTape() as tape:
            predictions = model(images, training=True)
            total_loss = tf.constant(0.0, dtype=tf.float32)
            batch_ciou = tf.constant(0.0, dtype=tf.float32)
            batch_obj = tf.constant(0.0, dtype=tf.float32)
            batch_cls = tf.constant(0.0, dtype=tf.float32)
            for pred, target, loss_fn in zip(predictions, (target_small, target_medium, target_large), loss_fns):
                loss_val, ciou_comp, obj_comp, cls_comp = loss_fn.compute_total_and_components(target, pred)
                loss_val = tf.reduce_mean(loss_val)
                ciou_comp = tf.reduce_mean(ciou_comp)
                obj_comp = tf.reduce_mean(obj_comp)
                cls_comp = tf.reduce_mean(cls_comp)
                total_loss += loss_val
                batch_ciou += ciou_comp
                batch_obj += obj_comp
                batch_cls += cls_comp

        gradients = tape.gradient(total_loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        train_loss_metric.update_state(total_loss)
        ciou_loss_metric.update_state(batch_ciou)
        obj_loss_metric.update_state(batch_obj)
        cls_loss_metric.update_state(batch_cls)
        return total_loss, batch_ciou, batch_obj, batch_cls

    start_epoch = int(ckpt.epoch.numpy())
    print(f"{datetime.now()}, start training from epoch {start_epoch + 1}.")


    best_loss = np.inf
    patience = 30
    patience_counter = 0
    for epoch in range(start_epoch, EPOCHS):
        train_loss_metric.reset_state()
        ciou_loss_metric.reset_state()
        obj_loss_metric.reset_state()
        cls_loss_metric.reset_state()

        # --- tqdm progress bar here ---
        progress = tqdm(dataset, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch")

        for step, (images, labels, counts) in enumerate(progress):
            target_small, target_medium, target_large = build_targets(labels, counts)
            loss_value, ciou_value, obj_value, cls_value = train_step(images, target_small, target_medium, target_large)

            if step % 10 == 0:
                current_lr = get_current_lr(optimizer)
                progress.set_postfix(
                    # loss=float(loss_value.numpy()),
                    # ciou=float(ciou_value.numpy()),
                    # obj=float(obj_value.numpy()),
                    # cls=float(cls_value.numpy()),
                    lr=float(current_lr.numpy()),
                )

        ckpt.epoch.assign(epoch + 1)
        epoch_loss = train_loss_metric.result().numpy()
        epoch_ciou = ciou_loss_metric.result().numpy()
        epoch_obj = obj_loss_metric.result().numpy()
        epoch_cls = cls_loss_metric.result().numpy()
        print(
            f"{datetime.now()}, Epoch {epoch + 1}: "
            f"loss {epoch_loss:.4f} | ciou {epoch_ciou:.4f} | obj {epoch_obj:.4f} | cls {epoch_cls:.4f}"
        )

        if(epoch & 1 or epoch == EPOCHS - 1):
            save_path = manager.save()
            print(f"Saved checkpoint for epoch {epoch + 1}: {save_path}")

        # Early stopping check
        if epoch_loss < best_loss - 1e-4:
            best_loss = epoch_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter > patience:
            print(f"Early stopping at epoch {epoch+1}")
            break


if __name__ == "__main__":
    main()


## inference.py

In [ ]:
"""
Single-image inference helper for the ConvNeXt-backed YOLOv4 model.

Adjust the constants near the bottom (checkpoint, images, thresholds),
then run `python inference.py` to produce annotated JPEGs.
"""

from __future__ import annotations

import os
from typing import List, Tuple

import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont

from constants import IMAGE_SIZE, NUM_CLASSES, ROOT, classes_name, VALID_IMAGE_NUMBER
from yolov4.model import build_yolov4, decode_predictions


ANCHOR_SCALE = IMAGE_SIZE / 448.0
ANCHOR_SETS = [
    [
        (12 * ANCHOR_SCALE, 16 * ANCHOR_SCALE),
        (19 * ANCHOR_SCALE, 36 * ANCHOR_SCALE),
        (40 * ANCHOR_SCALE, 28 * ANCHOR_SCALE),
    ],
    [
        (36 * ANCHOR_SCALE, 75 * ANCHOR_SCALE),
        (76 * ANCHOR_SCALE, 55 * ANCHOR_SCALE),
        (72 * ANCHOR_SCALE, 146 * ANCHOR_SCALE),
    ],
    [
        (142 * ANCHOR_SCALE, 110 * ANCHOR_SCALE),
        (192 * ANCHOR_SCALE, 243 * ANCHOR_SCALE),
        (459 * ANCHOR_SCALE, 401 * ANCHOR_SCALE),
    ],
]
STRIDES = (8, 16, 32)
XY_SCALES = (1.2, 1.1, 1.05)


def xywh_to_yxyx(xywh: tf.Tensor) -> tf.Tensor:
    """Convert `[xc, yc, w, h]` to `[ymin, xmin, ymax, xmax]`."""

    x, y, w, h = tf.split(xywh, 4, axis=-1)
    half_w = w / 2.0
    half_h = h / 2.0
    return tf.concat([y - half_h, x - half_w, y + half_h, x + half_w], axis=-1)


def load_and_preprocess(image_path: str) -> Tuple[tf.Tensor, Image.Image]:
    """Resize to training resolution and match the [-1, 1] scaling."""

    pil_image = Image.open(image_path).convert("RGB")
    np_image = np.asarray(pil_image, dtype=np.float32)
    tensor = tf.convert_to_tensor(np_image)
    tensor = tf.image.resize(tensor, [IMAGE_SIZE, IMAGE_SIZE])
    tensor = (tensor / 255.0) * 2.0 - 1.0
    tensor = tf.expand_dims(tensor, axis=0)
    return tensor, pil_image


def model_predictions(model: tf.keras.Model, image_tensor: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    """Run a forward pass and decode YOLO heads into absolute box coordinates."""

    predictions = model(image_tensor, training=False)
    batch_size = tf.shape(image_tensor)[0]
    all_boxes: List[tf.Tensor] = []
    all_scores: List[tf.Tensor] = []

    for pred, anchors, stride, xy_scale in zip(predictions, ANCHOR_SETS, STRIDES, XY_SCALES):
        anchors_tensor = tf.constant(anchors, dtype=tf.float32)
        bbox, objectness, class_logits, _ = decode_predictions(
            pred,
            anchors=anchors_tensor,
            stride=stride,
            xy_scale=xy_scale,
        )
        class_probs = tf.nn.sigmoid(class_logits)
        scores = objectness * class_probs
        boxes = xywh_to_yxyx(bbox) / float(IMAGE_SIZE)

        all_boxes.append(tf.reshape(boxes, (batch_size, -1, 1, 4)))
        all_scores.append(tf.reshape(scores, (batch_size, -1, NUM_CLASSES)))

    return tf.concat(all_boxes, axis=1), tf.concat(all_scores, axis=1)


def run_nms(
    boxes: tf.Tensor,
    scores: tf.Tensor,
    *,
    max_detections: int,
    iou_thresh: float,
    score_thresh: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, int]:
    """Apply class-aware NMS and return numpy arrays."""

    nms_output = tf.image.combined_non_max_suppression(
        boxes=boxes,
        scores=scores,
        max_output_size_per_class=max_detections,
        max_total_size=max_detections,
        iou_threshold=iou_thresh,
        score_threshold=score_thresh,
        clip_boxes=False,
    )

    return (
        nms_output.nmsed_boxes[0].numpy(),
        nms_output.nmsed_scores[0].numpy(),
        nms_output.nmsed_classes[0].numpy().astype(np.int32),
        int(nms_output.valid_detections[0].numpy()),
    )


def draw_detections(
    image: Image.Image,
    boxes: np.ndarray,
    scores: np.ndarray,
    classes: np.ndarray,
    valid: int,
    *,
    output_path: str,
) -> None:
    """Overlay bounding boxes and save the visualization."""

    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    width, height = image.size

    for idx in range(valid):
        ymin, xmin, ymax, xmax = boxes[idx]
        score = scores[idx]
        cls_id = classes[idx]

        xmin = float(np.clip(xmin, 0.0, 1.0)) * width
        xmax = float(np.clip(xmax, 0.0, 1.0)) * width
        ymin = float(np.clip(ymin, 0.0, 1.0)) * height
        ymax = float(np.clip(ymax, 0.0, 1.0)) * height

        label = classes_name[cls_id] if 0 <= cls_id < len(classes_name) else str(cls_id)
        text = f"{label}: {score:.2f}"

        draw.rectangle([(xmin, ymin), (xmax, ymax)], outline="red", width=2)
        if hasattr(draw, "textbbox"):
            text_box = draw.textbbox((xmin, ymin), text, font=font)
        else:
            tw, th = draw.textsize(text, font=font)
            text_box = (xmin, ymin, xmin + tw, ymin + th)
        draw.rectangle(text_box, fill="red")
        draw.text((text_box[0], text_box[1]), text, fill="white", font=font)

    image.save(output_path)
    print(f"Saved visualization to {output_path} ({valid} detections)")


def resolve_checkpoint(path: str) -> str:
    if os.path.isdir(path):
        latest = tf.train.latest_checkpoint(path)
        if latest:
            return latest
        raise FileNotFoundError(f"No checkpoints under {path}")
    if tf.io.gfile.exists(path + ".index"):
        return path
    raise FileNotFoundError(f"Checkpoint {path} not found")


def load_detector(checkpoint_path: str, *, force_cpu: bool) -> tf.keras.Model:
    """Build YOLOv4, restore weights, and optionally hide XPU devices."""

    if force_cpu:
        try:
            tf.config.experimental.set_visible_devices([], "XPU")
        except Exception:
            pass

    resolved = resolve_checkpoint(checkpoint_path)
    model = build_yolov4(
        (IMAGE_SIZE, IMAGE_SIZE, 3),
        num_classes=NUM_CLASSES,
        backbone_weights=None,
        backbone_trainable=False,
    )
    ckpt = tf.train.Checkpoint(model=model)
    ckpt.restore(resolved).expect_partial()
    print(f"Loaded weights from {resolved}")
    return model


def predict_one(
    model: tf.keras.Model,
    image_path: str,
    output_path: str,
    *,
    score_thresh: float,
    iou_thresh: float,
    max_detections: int,
) -> None:
    """Preprocess, run the detector, apply NMS, and save visualization."""

    if not os.path.exists(image_path):
        raise FileNotFoundError(image_path)

    image_tensor, pil_image = load_and_preprocess(image_path)
    boxes, scores = model_predictions(model, image_tensor)
    nms_boxes, nms_scores, nms_classes, valid = run_nms(
        boxes,
        scores,
        max_detections=max_detections,
        iou_thresh=iou_thresh,
        score_thresh=score_thresh,
    )

    if valid == 0:
        print(f"No detections above threshold for {image_path}")
        pil_image.save(output_path)
        return

    draw_detections(
        pil_image,
        nms_boxes,
        nms_scores,
        nms_classes,
        valid,
        output_path=output_path,
    )


# ------------------------------
# User-editable configuration
# ------------------------------
SAMPLE_IMAGES = [
    (
        os.path.join(ROOT, "VOCdevkit_train", "VOC2007", "JPEGImages", f"{idx:06d}.jpg"),
        f"./predict_img/prediction_{idx:06d}.jpg",
    )
    for idx in VALID_IMAGE_NUMBER
]

CONFIDENCE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.4
MAX_DETECTIONS = 50
CHECKPOINT_PATH = os.path.join("ckpts", "focal2", "ckpt-150")


def main() -> None:
    model = load_detector(CHECKPOINT_PATH, force_cpu=FORCE_ITEX_CPU)
    for image_path, output_path in SAMPLE_IMAGES:
        try:
            predict_one(
                model,
                image_path,
                output_path,
                score_thresh=CONFIDENCE_THRESHOLD,
                iou_thresh=IOU_THRESHOLD,
                max_detections=MAX_DETECTIONS,
            )
        except Exception as e:
            print(e)
            pass


if __name__ == "__main__":
    main()


## generate_predictions

In [ ]:
"""
Generate competition-format detections for the VOC test split.

This script reads `dataset/pascal_voc_testing_data.txt`, runs the ConvNeXt
YOLOv4 detector in batches, and writes `dataset/test_prediction.txt` with
lines formatted as:

    image_name xmin ymin xmax ymax class_id confidence
"""

from __future__ import annotations

import os
from typing import Iterable, List, Sequence, Tuple

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Set TensorFlow logging to only show errors

import numpy as np
import tensorflow as tf
from tqdm import tqdm

from constants import IMAGE_SIZE, ROOT, TEST_BATCH_SIZE, SCORE_THRESHOLD, IOU_THRESHOLD, MAX_DETECTIONS, TEST_LIST, TEST_IMAGE_DIR, CHECKPOINT_PATH, PREDICTION_OPATH, FORCE_ITEX_CPU
from inference import load_detector, model_predictions


GPU_CONFIG = 'GPU' # Change to 'GPU' if using NVIDIA GPUs
gpus = tf.config.experimental.list_physical_devices(GPU_CONFIG)
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        # Select GPU number 1
        tf.config.experimental.set_visible_devices(gpus[0], GPU_CONFIG)
        logical_gpus = tf.config.experimental.list_logical_devices(GPU_CONFIG)
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)


def read_image_list(list_path: str) -> List[str]:
    with open(list_path, "r", encoding="utf-8") as f:
        names = []
        for line in f:
            token = line.strip().split(" ")[0]
            if token:
                names.append(token)
    return names


def preprocess_image(image_path: str) -> Tuple[np.ndarray, float, float]:
    """Resize to IMAGE_SIZE and return numpy tensor plus original dims."""

    data = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(data, channels=3)
    height = float(tf.shape(image)[0].numpy())
    width = float(tf.shape(image)[1].numpy())
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = (image / 255.0) * 2.0 - 1.0
    return image.numpy(), height, width


def chunk(sequence: Sequence[str], size: int) -> Iterable[Sequence[str]]:
    for idx in range(0, len(sequence), size):
        yield sequence[idx : idx + size]


def batch_nms(
    boxes: tf.Tensor,
    scores: tf.Tensor,
    *,
    max_detections: int,
    iou_thresh: float,
    score_thresh: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    nms = tf.image.combined_non_max_suppression(
        boxes=boxes,
        scores=scores,
        max_output_size_per_class=max_detections,
        max_total_size=max_detections,
        iou_threshold=iou_thresh,
        score_threshold=score_thresh,
        clip_boxes=False,
    )
    return (
        nms.nmsed_boxes.numpy(),
        nms.nmsed_scores.numpy(),
        nms.nmsed_classes.numpy().astype(np.int32),
        nms.valid_detections.numpy().astype(np.int32),
    )


def write_predictions(
    image_names: Sequence[str],
    boxes: np.ndarray,
    scores: np.ndarray,
    classes: np.ndarray,
    valid: np.ndarray,
    heights: Sequence[float],
    widths: Sequence[float],
    *,
    file_handle,
) -> None:
    for idx, name in enumerate(image_names):
        det_count = int(valid[idx])
        if det_count == 0:
            continue

        height = heights[idx]
        width = widths[idx]

        for det in range(det_count):
            ymin, xmin, ymax, xmax = boxes[idx, det]
            score = float(scores[idx, det])
            cls_id = int(classes[idx, det])

            xmin_px = int(np.clip(np.round(xmin * width), 0, max(width - 1, 0)))
            xmax_px = int(np.clip(np.round(xmax * width), 0, max(width - 1, 0)))
            ymin_px = int(np.clip(np.round(ymin * height), 0, max(height - 1, 0)))
            ymax_px = int(np.clip(np.round(ymax * height), 0, max(height - 1, 0)))

            line = f"{name} {xmin_px} {ymin_px} {xmax_px} {ymax_px} {cls_id} {score:.6f}\n"
            file_handle.write(line)

def main() -> None:
    if not os.path.exists(TEST_LIST):
        raise FileNotFoundError(TEST_LIST)
    if not os.path.isdir(TEST_IMAGE_DIR):
        raise FileNotFoundError(TEST_IMAGE_DIR)

    image_names = read_image_list(TEST_LIST)
    if not image_names:
        raise RuntimeError("No test images found.")

    print(f"Found {len(image_names)} test images. Loading model...")
    model = load_detector(CHECKPOINT_PATH, force_cpu=FORCE_ITEX_CPU)

    os.makedirs(os.path.dirname(PREDICTION_OPATH), exist_ok=True)
    total_written = 0

    with open(PREDICTION_OPATH, "w", encoding="utf-8") as out_f:
        for batch_idx, batch_names in tqdm(
            enumerate(chunk(image_names, TEST_BATCH_SIZE)),
            total=(len(image_names) + TEST_BATCH_SIZE - 1) // TEST_BATCH_SIZE,
            desc="Processing batches"
        ):
            batch_images = []
            heights = []
            widths = []

            for name in batch_names:
                image_path = os.path.join(TEST_IMAGE_DIR, name)
                image_np, height, width = preprocess_image(image_path)
                batch_images.append(image_np)
                heights.append(height)
                widths.append(width)

            batch_tensor = tf.convert_to_tensor(np.stack(batch_images, axis=0), dtype=tf.float32)
            boxes, scores = model_predictions(model, batch_tensor)
            nms_boxes, nms_scores, nms_classes, valid = batch_nms(
                boxes,
                scores,
                max_detections=MAX_DETECTIONS,
                iou_thresh=IOU_THRESHOLD,
                score_thresh=SCORE_THRESHOLD,
            )

            write_predictions(
                batch_names,
                nms_boxes,
                nms_scores,
                nms_classes,
                valid,
                heights,
                widths,
                file_handle=out_f,
            )

            total_written += int(np.sum(valid))

    print(f"Wrote predictions for {len(image_names)} images to {PREDICTION_OPATH}")
    print(f"Total detections: {total_written}")


if __name__ == "__main__":
    main()
